In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)


# What computers see: images as numbers

A grayscale image can be stored as a 2D matrix. Each number represents pixel intensity. For many image datasets, pixel values range from 0 to 255.


In [ ]:
# A tiny 8 x 8 grayscale image represented as numbers
small_image = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 30, 80, 80, 30, 0, 0],
    [0, 30, 180, 255, 255, 180, 30, 0],
    [0, 80, 255, 120, 120, 255, 80, 0],
    [0, 80, 255, 120, 120, 255, 80, 0],
    [0, 30, 180, 255, 255, 180, 30, 0],
    [0, 0, 30, 80, 80, 30, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0]
])

plt.figure(figsize=(4, 4))
plt.imshow(small_image, cmap='gray', vmin=0, vmax=255)
plt.title('A tiny grayscale image')
plt.axis('off')
plt.show()

print('Shape:', small_image.shape)
print('Pixel matrix:')
print(small_image)


**Question 1.** What does a larger number mean in a grayscale image?

**Your answer:**  



---
# Handwritten digit images

MNIST images are grayscale handwritten digits. Each image has shape:

`height x width`

For MNIST, each image is 28 x 28. This matches the lecture example where a digit image is represented as 784 pixel values.


In [ ]:
# Load MNIST handwritten digit data. Each image is 28 x 28.
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

class_names = [str(i) for i in range(10)]

print('Training images shape:', x_train.shape)
print('Training labels shape:', y_train.shape)
print('Test images shape:', x_test.shape)
print('One image shape:', x_train[0].shape)


In [ ]:
# Visualize one handwritten digit image
idx = 0
img = x_train[idx]
label = y_train[idx]

plt.figure(figsize=(4, 4))
plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.title(f'Label: {label}')
plt.axis('off')
plt.show()

print('Image shape:', img.shape)
print('Minimum pixel value:', img.min())
print('Maximum pixel value:', img.max())


In [ ]:
# Zoom in and inspect pixel values
plt.figure(figsize=(6, 6))
plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.title('Handwritten digit as pixels')
plt.axis('off')

# Add pixel values on top of the image
for row in range(img.shape[0]):
    for col in range(img.shape[1]):
        if img[row, col] > 0:
            plt.text(col, row, str(img[row, col]), ha='center', va='center', fontsize=5)

plt.show()

print('A 10 x 10 section from the center of the image:')
print(img[9:19, 9:19])


---
# Convolution by hand

A convolution filter, also called a kernel, looks at a small local region of the image at a time. This helps CNNs learn local patterns such as edges, corners, and textures.

Below, we apply a simple vertical edge detector to a small matrix.


In [ ]:
# A simple 5 x 5 image matrix
image_matrix = np.array([
    [7, 2, 3, 3, 8],
    [4, 5, 3, 8, 4],
    [3, 3, 2, 8, 4],
    [2, 8, 7, 2, 7],
    [5, 4, 4, 5, 4]
])

# A 3 x 3 vertical edge filter
kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
])

print('Image matrix:')
print(image_matrix)
print('Kernel:')
print(kernel)


In [ ]:
def convolve2d_valid(image, kernel):
    # Apply valid 2D convolution/cross-correlation without padding.
    image_h, image_w = image.shape
    kernel_h, kernel_w = kernel.shape
    out_h = image_h - kernel_h + 1
    out_w = image_w - kernel_w + 1
    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            region = image[i:i+kernel_h, j:j+kernel_w]
            output[i, j] = np.sum(region * kernel)
    return output

feature_map = convolve2d_valid(image_matrix, kernel)
print('Feature map:')
print(feature_map)


In [ ]:
# Visualize the input and output
plt.figure(figsize=(8, 3))

plt.subplot(1, 2, 1)
plt.imshow(image_matrix, cmap='gray')
plt.title('Input matrix')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(feature_map, cmap='gray')
plt.title('Feature map after convolution')
plt.colorbar()

plt.show()


**Question 2.** Why does a convolution filter focus on local regions instead of connecting every pixel to every neuron?

**Your answer:**  



---
# Stride and padding

Stride controls how far the filter moves each step. Padding adds zeros around the image boundary so the filter can cover edge pixels and sometimes preserve output size.


In [ ]:
def convolve2d(image, kernel, stride=1, padding=0):
    # Apply 2D convolution/cross-correlation with stride and zero padding.
    if padding > 0:
        image = np.pad(image, pad_width=padding, mode='constant', constant_values=0)

    image_h, image_w = image.shape
    kernel_h, kernel_w = kernel.shape
    out_h = (image_h - kernel_h) // stride + 1
    out_w = (image_w - kernel_w) // stride + 1
    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            row = i * stride
            col = j * stride
            region = image[row:row+kernel_h, col:col+kernel_w]
            output[i, j] = np.sum(region * kernel)
    return output

output_stride1 = convolve2d(image_matrix, kernel, stride=1, padding=0)
output_stride2 = convolve2d(image_matrix, kernel, stride=2, padding=0)
output_padding1 = convolve2d(image_matrix, kernel, stride=1, padding=1)

print('No padding, stride = 1, shape:', output_stride1.shape)
print(output_stride1)

print('No padding, stride = 2, shape:', output_stride2.shape)
print(output_stride2)

print('Padding = 1, stride = 1, shape:', output_padding1.shape)
print(output_padding1)


**Question 3.** What happens to the output size when stride increases? What happens when padding is added?

**Your answer:**  



---
# Pooling

Pooling reduces the spatial size of a feature map. Max pooling keeps the largest value in each region. Average pooling keeps the average value.


In [ ]:
pool_input = np.array([
    [12, 20, 30, 0],
    [8, 12, 2, 0],
    [34, 70, 37, 4],
    [112, 100, 25, 12]
])

def max_pool2d(image, pool_size=2, stride=2):
    out_h = (image.shape[0] - pool_size) // stride + 1
    out_w = (image.shape[1] - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            row = i * stride
            col = j * stride
            region = image[row:row+pool_size, col:col+pool_size]
            output[i, j] = np.max(region)
    return output

def avg_pool2d(image, pool_size=2, stride=2):
    out_h = (image.shape[0] - pool_size) // stride + 1
    out_w = (image.shape[1] - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            row = i * stride
            col = j * stride
            region = image[row:row+pool_size, col:col+pool_size]
            output[i, j] = np.mean(region)
    return output

print('Input:')
print(pool_input)
print('Max pooling output:')
print(max_pool2d(pool_input))
print('Average pooling output:')
print(avg_pool2d(pool_input))


**Question 4.** In your own words, why do CNNs use pooling layers?

**Your answer:**  



---
# Flattening

After convolution and pooling, the feature maps are often flattened into a 1D vector before being passed to fully connected layers.


In [ ]:
pooled_feature_map = np.array([
    [6, 8],
    [4, 7]
])

flattened = pooled_feature_map.flatten()

print('Pooled feature map shape:', pooled_feature_map.shape)
print(pooled_feature_map)
print('Flattened vector shape:', flattened.shape)
print(flattened)


**Question 5.** Why do we flatten the feature maps before a dense layer?

**Your answer:**  



---
# Build a simple CNN for handwritten digit classification

We will use MNIST. To keep classwork fast, we use a smaller subset of the dataset.


In [ ]:
# Normalize images to the range 0 to 1 and add a channel dimension
# CNN layers expect images in the format: height x width x channels
x_train_small = x_train[:10000].astype('float32') / 255.0
y_train_small = y_train[:10000]

x_test_small = x_test[:2000].astype('float32') / 255.0
y_test_small = y_test[:2000]

x_train_small = np.expand_dims(x_train_small, axis=-1)
x_test_small = np.expand_dims(x_test_small, axis=-1)

print('Training subset shape:', x_train_small.shape)
print('Test subset shape:', x_test_small.shape)


In [ ]:
cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()


In [ ]:
history = cnn_model.fit(
    x_train_small,
    y_train_small,
    epochs=3,
    batch_size=64,
    validation_split=0.2
)


In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_small, y_test_small, verbose=0)
print('CNN test accuracy:', cnn_test_acc)


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history.history['accuracy'], label='Training accuracy')
plt.plot(history.history['val_accuracy'], label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('CNN training history')
plt.legend()
plt.show()


---
# Compare with a dense neural network

A dense neural network flattens the image immediately and does not explicitly use local spatial structure.


In [ ]:
dense_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

dense_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

dense_model.summary()


In [ ]:
dense_test_loss, dense_test_acc = dense_model.evaluate(x_test_small, y_test_small, verbose=0)
print('Dense neural network test accuracy:', dense_test_acc)
print('CNN test accuracy:', cnn_test_acc)


**Question 6.** Which model performs better on the test set? Why might CNNs work better for images?

**Your answer:**  



---
# Data augmentation

Data augmentation creates modified versions of training images, such as flipped or rotated images. This can help the model generalize better.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1)
])

sample = x_train_small[0:1]

plt.figure(figsize=(10, 4))
for i in range(8):
    augmented = data_augmentation(sample, training=True)
    plt.subplot(2, 4, i + 1)
    plt.imshow(augmented[0, :, :, 0], cmap='gray', vmin=0, vmax=1)
    plt.axis('off')
plt.suptitle('Augmented versions of the same handwritten digit')
plt.show()


---
# Transfer learning with VGG16

Transfer learning means using a model that has already learned useful image patterns from a large dataset.

A common example is **VGG16**, which was trained on ImageNet. VGG16 can recognize general visual features such as edges, curves, textures, and object parts. For a new image classification task, we can reuse the convolutional base of VGG16 and add a new classifier for our own classes.

In practice, VGG16 is more suitable for color image tasks, such as classifying cats and dogs, medical images, or manufacturing defect images. MNIST is very simple, so our small CNN is enough. The code below shows the basic transfer learning workflow.


In [ ]:
# Simple VGG16 transfer learning example
# This cell shows the structure of transfer learning.
# It may take a little time the first time because the VGG16 weights need to download.

vgg_base = keras.applications.VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze the pre-trained convolutional base
vgg_base.trainable = False

vgg_transfer_model = keras.Sequential([
    vgg_base,
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(2, activation='softmax')  # Example: two classes, such as cat vs. dog
])

vgg_transfer_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

vgg_transfer_model.summary()



The important idea is that the early VGG16 layers are reused, while the final classification layer is changed for the new task.



In [ ]:
history = vgg_transfer_model.fit(
    x_train_small,
    y_train_small,
    epochs=3,
    batch_size=64,
    validation_split=0.2
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_small, y_test_small, verbose=0)
print('CNN test accuracy:', cnn_test_acc)

---
# Question 7

Answer these before submitting:

1. How is a handwritten digit image stored as numbers?
2. What does a convolution filter do?
3. What is the purpose of max pooling?
4. Why is CNN usually better than a regular dense neural network for images?
5. What is one situation where transfer learning would be useful?

**Your answers:**  


---
# Task: Mini coding assignment: Change the CNN layers

In this short assignment, you will change the CNN model from the earlier section and compare the performance.



Create a new model called `cnn_model_2`.

Make **one simple change** to the original CNN:

* Add one more convolution layer, or
* Remove one convolution layer, or
* Change the number of filters in a convolution layer

Then train the model and compare its test accuracy with the original CNN.
